# Подбор импьютера и регрессора

Сравниваем пары по среднему RMSE на пяти фолдах обучающей выборки. Итоговые метрики победителя считаем на отложенном тесте. Модели используют параметры из `build_model()`. Для запуска XGBoost на GPU включите `USE_GPU`.

In [1]:
from functools import partial
from pathlib import Path

import cupy as cp
import pandas as pd
from category_encoders import OneHotEncoder as NaNOneHotEncoder
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from imputer.imputer import EMImputer, GMMImputer
from models import gradient_boosting, knn, linear, random_forest, regression_tree, xgboost

In [2]:
DATA_PATH = Path("data/data.csv")
TARGET = "kosten_je_flaeche"
RANDOM_STATE = 42
USE_GPU = False

numeric = ["flaeche", "baujahr", "etage", "anz_zimmer"]
categorical = ["objekttyp_fein", "aus_klassen5"]
flags = """
aus_boden_kunststoff_janein aus_boden_laminat_janein aus_boden_linoleum_janein
aus_boden_marmor_janein aus_balkon_janein aus_boden_estrich_janein
aus_terrasse_janein aus_boden_fliesen_janein aus_boden_terrakotta_janein
aus_boden_parkett_janein aus_heiz_fernwaerme_janein aus_heiz_fussboden_janein
aus_kueche_einbau_janein aus_kueche_offen_janein oeig_neubau_janein
aus_boden_teppich_janein oeig_keller_janein aadr_gewerblich_janein
aus_alarmanlage_janein aus_abstellraum_janein anzart_indexmiete_janein
anzart_nachmieter_janein anzart_genossenschaft_janein aus_bad_dusche_janein
aus_bad_mit_fenster_janein aus_bad_gaestewc_janein aus_barrierefrei_janein
aus_bad_wanne_janein energie_niedrig_janein aus_pool_janein aus_sauna_janein
aus_parken_tiefgarage_janein aus_hauswirtschaftsraum_janein aus_garten_janein
aus_kamin_janein aus_lastenaufzug_janein aus_lift_janein aus_parken_garage_janein
aus_parken_janein aus_parken_ladestation_janein oeig_loggia_janein oeig_galerie_janein
zust_gepflegt_janein zust_projektiert_janein zust_sanierungsbedarf_janein
""".split()

features = numeric + categorical + flags
df = pd.read_csv(DATA_PATH, usecols=features + [TARGET], dtype=dict.fromkeys(flags, "boolean"))
for col in flags:
    if set(df[col].dropna().unique()) == {True}:
        df[col] = df[col].fillna(False)

df[flags] = df[flags].astype(float)
df = df.dropna(subset=[TARGET])
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[TARGET], test_size=0.2, random_state=RANDOM_STATE
)

In [3]:
imputers = [
    ("GMMImputer (k=3)", GMMImputer(n_components=3)),
    ("SimpleImputer (mean)", SimpleImputer(strategy="mean")),
    ("SimpleImputer (median)", SimpleImputer(strategy="median")),
    ("KNNImputer (k=5)", KNNImputer(n_neighbors=5)),
    ("IterativeImputer", IterativeImputer(random_state=RANDOM_STATE, max_iter=20)),
]

models = [
    ("Linear regression", linear.build_model, False),
    ("Regression tree", regression_tree.build_model, False),
    ("Random forest", random_forest.build_model, False),
    ("Gradient boosting", gradient_boosting.build_model, False),
    ("KNN", knn.build_model, False),
    ("XGBoost (GPU)", xgboost.build_model, True)
]


In [4]:
def prepare_features(imputer, X_fit, X_eval):
    encoder = NaNOneHotEncoder(cols=categorical, handle_missing="return_nan")
    X_fit_encoded = encoder.fit_transform(X_fit)
    X_eval_encoded = encoder.transform(X_eval)

    fitted = clone(imputer).fit(X_fit_encoded)
    return fitted.transform(X_fit_encoded), fitted.transform(X_eval_encoded)


def fit_and_score(model_spec, X_fit, X_eval, y_fit, y_eval):
    _, build_model, use_gpu = model_spec
    model = build_model()
    if use_gpu:
        X_fit = cp.asarray(X_fit, dtype=cp.float32)
        y_fit = cp.asarray(y_fit.to_numpy(), dtype=cp.float32)
        X_eval = cp.asarray(X_eval, dtype=cp.float32)

    model.fit(X_fit, y_fit)
    predictions = model.predict(X_eval)
    if use_gpu:
        predictions = cp.asnumpy(predictions)

    return {
        "rmse": mean_squared_error(y_eval, predictions) ** 0.5,
        "mae": mean_absolute_error(y_eval, predictions),
        "r2": r2_score(y_eval, predictions),
    }

In [ ]:
%%time
from utils import run_grid_search

ranking = run_grid_search(
    X_train,
    y_train,
    imputers=imputers,
    models=models,
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    prepare_features=prepare_features,
    fit_and_score=fit_and_score,
)
display(ranking)

'14/175 (8%) | Running: GMMImputer (k=3) | Fold 3/5 | Imputation | 114.9s'

/home/mzarkov/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but GMMImputer was fitted with feature names
  warnings.warn(
/home/mzarkov/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but GMMImputer was fitted with feature names
  warnings.warn(
/home/mzarkov/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but GMMImputer was fitted with feature names
  warnings.warn(
/home/mzarkov/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but GMMImputer was fitted with feature names
  warnings.warn(


In [ ]:
winner = ranking.iloc[0]
best_imputer = dict(imputers)[winner["imputer"]]
best_model_spec = next(spec for spec in models if spec[0] == winner["model"])

X_fit, X_eval = prepare_features(best_imputer, X_train, X_test)
test_scores = fit_and_score(best_model_spec, X_fit, X_eval, y_train, y_test)
print(f"Лучшая пара по CV RMSE — {winner['imputer']} × {winner['model']}")
display(pd.DataFrame([test_scores], index=["test"]))